# Step 4. Clustering Model Selection

이 노트북의 목적은 Step 3.5에서 생성한 최종 입력 데이터를 기반으로 여러 clustering 후보를 비교하고, 이후 Step 4.5 Visualization 및 Step 5 Segment Profiling에서 사용할 **최종 clustering model key**를 선택하는 것이다.

이 단계에서는 시각화를 거의 수행하지 않는다. 핵심은 다음이다.

1. `BankChurners_cluster_input.csv`와 `BankChurners_profile_table.csv`를 불러온다.
2. 여러 feature set을 정의한다.
3. K-Means, Gaussian Mixture Model, Agglomerative Clustering을 비교한다.
4. k=4~6 범위에서 후보를 평가한다.
5. 내부 군집 지표와 사후 해석 지표를 함께 저장한다.
6. 최종 선택 모델을 `selected_model.json`과 `selected_cluster_labels.csv`로 저장한다.

> 주의: `Attrition_Flag`는 clustering input에 사용하지 않는다. 이탈률은 clustering 이후 사후 평가 및 segment profiling을 위한 지표로만 사용한다.

In [ ]:
# ============================================================
# Step 4. Clustering Model Selection
# ============================================================

from pathlib import Path
import sys
import json
import pickle
import warnings

import numpy as np
import pandas as pd

from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
)
from sklearn.exceptions import ConvergenceWarning
import joblib

warnings.filterwarnings("ignore", category=ConvergenceWarning)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

## 1. 프로젝트 루트 및 공통 함수 로드

노트북을 `notebooks/` 폴더에서 실행하든 프로젝트 루트에서 실행하든 동일하게 동작하도록 project root를 탐색한다.  
또한 `src/common.py`가 존재하면 공통 함수를 사용하고, 없거나 함수명이 다른 경우를 대비해 fallback 함수를 정의한다.

In [ ]:
# ------------------------------------------------------------
# 1. Project root detection and common utilities
# ------------------------------------------------------------

def find_project_root():
    current = Path.cwd().resolve()
    candidates = [current] + list(current.parents)

    for path in candidates:
        if (path / "data").exists() and (path / "src").exists():
            return path

    for path in candidates:
        if (path / "data").exists():
            return path

    return current


ROOT = find_project_root()
sys.path.insert(0, str(ROOT))

print("Project root:", ROOT)

# common.py 함수 import 시도
try:
    from src.common import get_project_root, read_csv_safely, configure_korean_font, save_json
    ROOT = get_project_root()
    print("Loaded utilities from src.common")
except Exception as exc:
    print("src.common import failed or incomplete. Using fallback utilities.")
    print("Reason:", exc)

    def read_csv_safely(path, **kwargs):
        encodings = ("utf-8-sig", "utf-8", "cp949", "euc-kr")
        last_error = None
        for enc in encodings:
            try:
                return pd.read_csv(path, encoding=enc, **kwargs)
            except UnicodeDecodeError as e:
                last_error = e
        raise last_error

    def save_json(data, path):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=4)

    def configure_korean_font(verbose=True):
        return None

# 시각화가 많지는 않지만, 한글 폰트 설정을 미리 수행한다.
try:
    configure_korean_font(verbose=True)
except Exception as exc:
    print("Korean font configuration skipped:", exc)

## 2. 입력 및 출력 경로 설정

Step 4의 입력은 Step 3.5에서 생성된 `data/processed/`의 두 파일이다.

- `BankChurners_cluster_input.csv`: 실제 clustering 알고리즘에 들어가는 feature matrix
- `BankChurners_profile_table.csv`: clustering 이후 이탈률 및 위험 신호를 사후 평가하기 위한 table

Step 4의 결과는 `data/analysis/clustering_results/`에 저장한다.

In [ ]:
# ------------------------------------------------------------
# 2. Paths
# ------------------------------------------------------------

processed_dir = ROOT / "data" / "processed"
output_dir = ROOT / "data" / "analysis" / "clustering_results"
output_dir.mkdir(parents=True, exist_ok=True)

cluster_input_path = processed_dir / "BankChurners_cluster_input.csv"
profile_table_path = processed_dir / "BankChurners_profile_table.csv"
feature_info_path = processed_dir / "cluster_feature_info.json"

if not cluster_input_path.exists():
    raise FileNotFoundError(
        f"Clustering input file not found: {cluster_input_path}\n"
        "먼저 notebooks/03_5_further_preprocessing.ipynb를 실행해야 합니다."
    )

if not profile_table_path.exists():
    raise FileNotFoundError(
        f"Profile table file not found: {profile_table_path}\n"
        "먼저 notebooks/03_5_further_preprocessing.ipynb를 실행해야 합니다."
    )

print("cluster_input_path:", cluster_input_path.relative_to(ROOT))
print("profile_table_path:", profile_table_path.relative_to(ROOT))
print("output_dir:", output_dir.relative_to(ROOT))

## 3. 데이터 로드 및 기본 검증

`Row_ID`는 `cluster_input`과 `profile_table`을 연결하기 위한 key이다.  
두 테이블의 행 수와 `Row_ID` 정렬이 동일해야 이후 cluster label을 안전하게 붙일 수 있다.

In [ ]:
# ------------------------------------------------------------
# 3. Load data and validate
# ------------------------------------------------------------

df_cluster = read_csv_safely(cluster_input_path)
df_profile = read_csv_safely(profile_table_path)

print("df_cluster shape:", df_cluster.shape)
print("df_profile shape:", df_profile.shape)

if "Row_ID" not in df_cluster.columns:
    raise ValueError("df_cluster에 Row_ID 컬럼이 없습니다.")

if "Row_ID" not in df_profile.columns:
    raise ValueError("df_profile에 Row_ID 컬럼이 없습니다.")

assert len(df_cluster) == len(df_profile), "df_cluster와 df_profile의 row 수가 다릅니다."
assert df_cluster["Row_ID"].equals(df_profile["Row_ID"]), "df_cluster와 df_profile의 Row_ID 순서가 다릅니다."

missing_cluster_values = int(df_cluster.isnull().sum().sum())
missing_profile_values = int(df_profile.isnull().sum().sum())

print("Missing values in df_cluster:", missing_cluster_values)
print("Missing values in df_profile:", missing_profile_values)

if missing_cluster_values > 0:
    raise ValueError("df_cluster에 결측치가 있습니다. Step 3.5 전처리를 확인하세요.")

display(df_cluster.head())
display(df_profile.head())

## 4. Clustering에 사용하면 안 되는 컬럼 검증

Step 4의 clustering input에는 이탈 여부, 인구통계 변수, label-encoded 범주형 변수가 들어가면 안 된다.  
현재 `df_cluster`는 Step 3.5에서 만들어진 행동 축만 포함해야 한다.

In [ ]:
# ------------------------------------------------------------
# 4. Forbidden columns check
# ------------------------------------------------------------

forbidden_cols = [
    "Attrition_Flag",
    "Attrition_Label",
    "Gender",
    "Gender_Label",
    "Education_Level",
    "Education_Label",
    "Marital_Status",
    "Marital_Label",
    "Income_Category",
    "Income_Label",
    "Card_Category",
    "Card_Label",
    "Customer_Age",
    "Dependent_count",
]

included_forbidden = [col for col in forbidden_cols if col in df_cluster.columns]

if included_forbidden:
    raise ValueError(f"clustering input에 들어가면 안 되는 컬럼이 포함되어 있습니다: {included_forbidden}")

print("Forbidden column check passed.")
print("Available clustering columns:")
print([col for col in df_cluster.columns if col != "Row_ID"])

## 5. Feature Set 정의

세 가지 feature set을 비교한다.

1. `axis_7`: Step 3.5에서 만든 7개 행동 축 전체
2. `axis_6_without_capacity`: 신용한도 규모(`Credit_Capacity_Score`)를 제외한 6개 행동 축
3. `weighted_axis_7`: 고객 유지 전략과 직접 연결되는 축에 약한 가중치를 부여한 7개 축

최종 선택은 단순 내부 지표만이 아니라 cluster size balance, 이탈률 차이, high-risk signal 차이, 해석 가능성을 함께 고려한다.

In [ ]:
# ------------------------------------------------------------
# 5. Define feature sets
# ------------------------------------------------------------

axis_7_features = [
    "Activity_Score",
    "Decline_Risk_Score",
    "Inactivity_Score",
    "Relationship_Strength_Score",
    "Contact_Burden_Score",
    "Credit_Capacity_Score",
    "Credit_Utilization_Score",
]

axis_6_without_capacity_features = [
    "Activity_Score",
    "Decline_Risk_Score",
    "Inactivity_Score",
    "Relationship_Strength_Score",
    "Contact_Burden_Score",
    "Credit_Utilization_Score",
]

feature_sets = {
    "axis_7": axis_7_features,
    "axis_6_without_capacity": axis_6_without_capacity_features,
}

feature_weights = {
    "Activity_Score": 1.2,
    "Decline_Risk_Score": 1.4,
    "Inactivity_Score": 1.2,
    "Relationship_Strength_Score": 1.0,
    "Contact_Burden_Score": 1.1,
    "Credit_Capacity_Score": 0.7,
    "Credit_Utilization_Score": 1.0,
}

# 필수 feature 존재 여부 확인
all_required_features = sorted(set(axis_7_features))
missing_features = [col for col in all_required_features if col not in df_cluster.columns]

if missing_features:
    raise ValueError(f"df_cluster에 필요한 feature가 없습니다: {missing_features}")

print("Feature sets are ready.")
for name, cols in feature_sets.items():
    print(f"{name}: {cols}")

print("weighted_axis_7 weights:")
feature_weights

## 6. 평가 함수 정의

Clustering 후보는 두 종류의 기준으로 평가한다.

### 내부 군집 지표
- Silhouette Score: 높을수록 좋음
- Davies-Bouldin Index: 낮을수록 좋음
- Calinski-Harabasz Score: 높을수록 좋음

### 사후 해석 지표
- cluster size balance
- segment별 이탈률 차이
- segment별 high-risk signal 차이

사후 해석 지표는 clustering에 사용하지 않은 profile table을 통해 계산한다.

In [ ]:
# ------------------------------------------------------------
# 6. Evaluation functions
# ------------------------------------------------------------

def apply_weighting(X: pd.DataFrame, weights: dict) -> pd.DataFrame:
    X_weighted = X.copy()
    for col, weight in weights.items():
        if col in X_weighted.columns:
            X_weighted[col] = X_weighted[col] * weight
    return X_weighted


def evaluate_internal_metrics(X: pd.DataFrame, labels: np.ndarray) -> dict:
    unique_labels = np.unique(labels)
    n_clusters = len(unique_labels)

    if n_clusters < 2:
        return {
            "silhouette": np.nan,
            "davies_bouldin": np.nan,
            "calinski_harabasz": np.nan,
        }

    return {
        "silhouette": float(silhouette_score(X, labels)),
        "davies_bouldin": float(davies_bouldin_score(X, labels)),
        "calinski_harabasz": float(calinski_harabasz_score(X, labels)),
    }


def evaluate_cluster_size(labels: np.ndarray) -> dict:
    counts = pd.Series(labels).value_counts().sort_index()
    ratios = counts / counts.sum()

    return {
        "min_cluster_size": int(counts.min()),
        "max_cluster_size": int(counts.max()),
        "min_cluster_ratio": float(ratios.min()),
        "max_cluster_ratio": float(ratios.max()),
        "cluster_size_ratio_gap": float(ratios.max() - ratios.min()),
    }


def evaluate_profile_metrics(df_profile: pd.DataFrame, labels: np.ndarray) -> dict:
    temp = df_profile.copy()
    temp["Cluster"] = labels

    group = temp.groupby("Cluster")

    profile_metrics = {}

    if "Attrition_Flag" in temp.columns:
        attrition_rates = group["Attrition_Flag"].mean()
        profile_metrics.update({
            "attrition_rate_min": float(attrition_rates.min()),
            "attrition_rate_max": float(attrition_rates.max()),
            "attrition_rate_range": float(attrition_rates.max() - attrition_rates.min()),
            "attrition_rate_std": float(attrition_rates.std(ddof=0)),
        })

    if "High_Risk_Signal_Count" in temp.columns:
        risk_means = group["High_Risk_Signal_Count"].mean()
        profile_metrics.update({
            "high_risk_signal_min": float(risk_means.min()),
            "high_risk_signal_max": float(risk_means.max()),
            "high_risk_signal_range": float(risk_means.max() - risk_means.min()),
            "high_risk_signal_std": float(risk_means.std(ddof=0)),
        })

    return profile_metrics


def make_profile_summary(df_profile: pd.DataFrame, labels: np.ndarray, candidate_key: str) -> pd.DataFrame:
    temp = df_profile.copy()
    temp["Cluster"] = labels

    agg_dict = {
        "Row_ID": "count",
    }

    optional_numeric_cols = [
        "Attrition_Flag",
        "Total_Trans_Ct",
        "Total_Trans_Amt",
        "Months_Inactive_12_mon",
        "Contacts_Count_12_mon",
        "Total_Relationship_Count",
        "Months_on_book",
        "Avg_Utilization_Ratio",
        "Avg_Ticket_Size",
        "Inactive_Ratio_12m",
        "Contact_Rate_12m",
        "No_Revolving_Bal_Flag",
        "Low_Relationship_Flag",
        "High_Inactive_Flag",
        "Low_Ct_Change_Flag",
        "Low_Amt_Change_Flag",
        "High_Contact_Flag",
        "Low_Utilization_Flag",
        "High_Risk_Signal_Count",
    ]

    for col in optional_numeric_cols:
        if col in temp.columns:
            agg_dict[col] = "mean"

    summary = temp.groupby("Cluster").agg(agg_dict).reset_index()

    rename_map = {
        "Row_ID": "n_customers",
        "Attrition_Flag": "attrition_rate",
        "Total_Trans_Ct": "mean_total_trans_ct",
        "Total_Trans_Amt": "mean_total_trans_amt",
        "Months_Inactive_12_mon": "mean_months_inactive",
        "Contacts_Count_12_mon": "mean_contacts_count",
        "Total_Relationship_Count": "mean_relationship_count",
        "Months_on_book": "mean_months_on_book",
        "Avg_Utilization_Ratio": "mean_avg_utilization_ratio",
    }

    summary = summary.rename(columns=rename_map)
    summary["candidate_key"] = candidate_key
    summary["cluster_ratio"] = summary["n_customers"] / summary["n_customers"].sum()

    ordered_cols = ["candidate_key", "Cluster"] + [
        col for col in summary.columns if col not in ["candidate_key", "Cluster"]
    ]

    return summary[ordered_cols]


print("Evaluation functions are ready.")

## 7. Clustering 후보 실행

아래 후보를 비교한다.

- K-Means: `axis_7`, `axis_6_without_capacity`, `weighted_axis_7`
- GMM: `axis_7`, `axis_6_without_capacity`
- Agglomerative Clustering: `axis_7`, `axis_6_without_capacity`

클러스터 수는 계획서 기준에 맞춰 `k=4,5,6`을 비교한다.

In [ ]:
# ------------------------------------------------------------
# 7. Run clustering candidates
# ------------------------------------------------------------

RANDOM_STATE = 42
K_RANGE = range(4, 7)

results = []
all_labels = {}
profile_summaries = {}
candidate_feature_matrices = {}

def add_candidate_result(key, feature_set_name, algorithm_name, k, X, labels):
    internal_metrics = evaluate_internal_metrics(X, labels)
    size_metrics = evaluate_cluster_size(labels)
    profile_metrics = evaluate_profile_metrics(df_profile, labels)

    all_labels[key] = labels
    candidate_feature_matrices[key] = X
    profile_summaries[key] = make_profile_summary(df_profile, labels, key)

    results.append({
        "key": key,
        "feature_set": feature_set_name,
        "algorithm": algorithm_name,
        "k": int(k),
        **internal_metrics,
        **size_metrics,
        **profile_metrics,
    })


# 7-1. axis_7, axis_6_without_capacity candidates
for feature_set_name, cols in feature_sets.items():
    X = df_cluster[cols].copy()

    for k in K_RANGE:
        # K-Means
        algorithm_name = "KMeans"
        model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=30)
        labels = model.fit_predict(X)

        key = f"{feature_set_name}_{algorithm_name}_k{k}"
        add_candidate_result(key, feature_set_name, algorithm_name, k, X, labels)

        # GMM
        algorithm_name = "GMM"
        model = GaussianMixture(
            n_components=k,
            covariance_type="full",
            random_state=RANDOM_STATE,
            n_init=10
        )
        labels = model.fit_predict(X)

        key = f"{feature_set_name}_{algorithm_name}_k{k}"
        add_candidate_result(key, feature_set_name, algorithm_name, k, X, labels)

        # Agglomerative
        algorithm_name = "Agglomerative_Ward"
        model = AgglomerativeClustering(n_clusters=k, linkage="ward")
        labels = model.fit_predict(X)

        key = f"{feature_set_name}_{algorithm_name}_k{k}"
        add_candidate_result(key, feature_set_name, algorithm_name, k, X, labels)


# 7-2. Weighted K-Means candidates
X_weighted = apply_weighting(df_cluster[axis_7_features].copy(), feature_weights)

for k in K_RANGE:
    algorithm_name = "Weighted_KMeans"
    model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=30)
    labels = model.fit_predict(X_weighted)

    key = f"weighted_axis_7_{algorithm_name}_k{k}"
    add_candidate_result(key, "weighted_axis_7", algorithm_name, k, X_weighted, labels)


result_df = pd.DataFrame(results)

print("Number of candidates:", len(result_df))
display(result_df.head())

## 8. 후보 비교표 생성

내부 지표와 사후 평가 지표를 하나의 table로 정리한다.  
이 단계에서 이탈률과 high-risk signal은 **모델 학습에 사용된 것이 아니라**, cluster가 실제로 의미 있는 고객 위험 차이를 반영하는지 확인하기 위한 사후 평가 기준이다.

In [ ]:
# ------------------------------------------------------------
# 8. Candidate comparison
# ------------------------------------------------------------

# 내부 지표 중심 정렬
internal_sorted = result_df.sort_values(
    by=["silhouette", "davies_bouldin", "calinski_harabasz"],
    ascending=[False, True, False]
).reset_index(drop=True)

# profile 평가까지 포함한 final comparison
final_compare = result_df.copy()

# 해석 가능성을 위한 간단한 composite score 생성
# - silhouette은 높을수록 좋음
# - attrition/high-risk range는 클수록 segment 차이가 잘 드러남
# - cluster size gap은 작을수록 좋음
# 아래 점수는 자동 선택을 위한 참고용이며, 최종 선택은 해석 가능성을 함께 고려한다.
def minmax_scale(s):
    s = s.astype(float)
    if s.max() == s.min():
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - s.min()) / (s.max() - s.min())

final_compare["silhouette_scaled"] = minmax_scale(final_compare["silhouette"])
final_compare["attrition_range_scaled"] = minmax_scale(final_compare.get("attrition_rate_range", pd.Series(0, index=final_compare.index)))
final_compare["risk_range_scaled"] = minmax_scale(final_compare.get("high_risk_signal_range", pd.Series(0, index=final_compare.index)))
final_compare["size_gap_scaled"] = minmax_scale(final_compare["cluster_size_ratio_gap"])

final_compare["selection_score"] = (
    0.35 * final_compare["silhouette_scaled"]
    + 0.25 * final_compare["attrition_range_scaled"]
    + 0.20 * final_compare["risk_range_scaled"]
    - 0.20 * final_compare["size_gap_scaled"]
)

final_compare_sorted = final_compare.sort_values(
    by=["selection_score", "silhouette", "attrition_rate_range", "high_risk_signal_range"],
    ascending=[False, False, False, False]
).reset_index(drop=True)

display_cols = [
    "key", "feature_set", "algorithm", "k",
    "silhouette", "davies_bouldin", "calinski_harabasz",
    "min_cluster_ratio", "max_cluster_ratio", "cluster_size_ratio_gap",
    "attrition_rate_range", "high_risk_signal_range",
    "selection_score"
]

display(final_compare_sorted[display_cols].head(15))

## 9. 최종 모델 선택

이전 실험 기준으로는 `axis_6_without_capacity_KMeans_k6`가 가장 균형적인 후보였다.

선택 이유는 다음과 같다.

1. 신용한도 규모(`Credit_Capacity_Score`)를 제외해 실제 이용 행동 중심의 segment를 만든다.
2. K-Means는 설명 가능성이 높고 제출/발표에 적합하다.
3. k=6은 계획서의 예상 segment 구조와 잘 맞는다.
4. 내부 지표가 최상위는 아니더라도 cluster size, 이탈률 차이, high-risk signal 차이, 해석 가능성이 균형적이다.

단, 실행 결과를 보고 다른 후보를 선택하고 싶다면 아래 `SELECTED_KEY`만 수정하면 된다.

In [ ]:
# ------------------------------------------------------------
# 9. Select final model
# ------------------------------------------------------------

# 기본 선택 후보
SELECTED_KEY = "axis_6_without_capacity_KMeans_k6"

if SELECTED_KEY not in all_labels:
    raise ValueError(f"SELECTED_KEY가 후보 목록에 없습니다: {SELECTED_KEY}")

selected_labels = all_labels[SELECTED_KEY]

selected_row = final_compare[final_compare["key"] == SELECTED_KEY].iloc[0].to_dict()
selected_profile_summary = profile_summaries[SELECTED_KEY].copy()

print("Selected model key:", SELECTED_KEY)
display(pd.DataFrame([selected_row])[display_cols])
display(selected_profile_summary)

## 10. 최종 선택 모델 재학습 및 저장

후속 단계에서 label만 사용해도 되지만, 재현성을 위해 선택된 K-Means 모델도 `joblib`으로 저장한다.  
현재 기본 선택 모델은 K-Means이므로 `predict()`를 이용해 향후 신규 고객의 segment를 배정할 수 있다.

In [ ]:
# ------------------------------------------------------------
# 10. Refit selected model and save if possible
# ------------------------------------------------------------

selected_model = None
selected_features = None
selected_X = None

if SELECTED_KEY.startswith("axis_6_without_capacity_KMeans"):
    selected_features = axis_6_without_capacity_features
    selected_X = df_cluster[selected_features].copy()
    selected_k = int(SELECTED_KEY.split("_k")[-1])
    selected_model = KMeans(n_clusters=selected_k, random_state=RANDOM_STATE, n_init=30)
    selected_model.fit(selected_X)

elif SELECTED_KEY.startswith("axis_7_KMeans"):
    selected_features = axis_7_features
    selected_X = df_cluster[selected_features].copy()
    selected_k = int(SELECTED_KEY.split("_k")[-1])
    selected_model = KMeans(n_clusters=selected_k, random_state=RANDOM_STATE, n_init=30)
    selected_model.fit(selected_X)

elif SELECTED_KEY.startswith("weighted_axis_7_Weighted_KMeans"):
    selected_features = axis_7_features
    selected_X = apply_weighting(df_cluster[selected_features].copy(), feature_weights)
    selected_k = int(SELECTED_KEY.split("_k")[-1])
    selected_model = KMeans(n_clusters=selected_k, random_state=RANDOM_STATE, n_init=30)
    selected_model.fit(selected_X)

else:
    print("선택된 모델은 재학습 저장 로직이 정의되지 않았습니다. label만 저장합니다.")

if selected_model is not None:
    selected_model_path = output_dir / "selected_clustering_model.joblib"
    joblib.dump(selected_model, selected_model_path)
    print("Selected model saved:", selected_model_path.relative_to(ROOT))
else:
    selected_model_path = None

## 11. 결과 파일 저장

Step 4 결과는 후속 Step 4.5와 Step 5에서 그대로 사용된다.

핵심 출력 파일은 다음이다.

- `03_final_model_comparison_sorted.csv`: 후보 모델 비교 결과
- `05_all_candidate_cluster_labels.csv`: 모든 후보의 cluster label
- `09_selected_model.json`: 최종 선택 모델 정보
- `10_selected_cluster_labels.csv`: 최종 모델의 cluster label
- `11_df_cluster_input_used.csv`: Step4에서 사용한 clustering input
- `12_df_profile_table_used.csv`: Step4에서 사용한 profile table

In [ ]:
# ------------------------------------------------------------
# 11. Save Step 4 outputs
# ------------------------------------------------------------

# 11-1. Candidate metrics
result_df.to_csv(
    output_dir / "01_internal_clustering_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

internal_sorted.to_csv(
    output_dir / "02_internal_clustering_metrics_sorted.csv",
    index=False,
    encoding="utf-8-sig"
)

final_compare.to_csv(
    output_dir / "03_final_model_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

final_compare_sorted.to_csv(
    output_dir / "04_final_model_comparison_sorted.csv",
    index=False,
    encoding="utf-8-sig"
)

final_compare_sorted.head(20).to_csv(
    output_dir / "04_top20_candidate_models.csv",
    index=False,
    encoding="utf-8-sig"
)

# 11-2. Candidate labels
label_df = pd.DataFrame({"Row_ID": df_cluster["Row_ID"].values})

for key, labels in all_labels.items():
    label_df[key] = labels

label_df.to_csv(
    output_dir / "05_all_candidate_cluster_labels.csv",
    index=False,
    encoding="utf-8-sig"
)

with open(output_dir / "05_all_candidate_cluster_labels.pkl", "wb") as f:
    pickle.dump(all_labels, f)

# 11-3. Candidate profile summaries
profile_summary_long = pd.concat(profile_summaries.values(), ignore_index=True)

profile_summary_long.to_csv(
    output_dir / "06_all_candidate_profile_summary_long.csv",
    index=False,
    encoding="utf-8-sig"
)

top10_keys = final_compare_sorted["key"].head(10).tolist()

top10_profile_summary = profile_summary_long[
    profile_summary_long["candidate_key"].isin(top10_keys)
].copy()

top10_profile_summary.to_csv(
    output_dir / "07_top10_candidate_profile_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

label_df[["Row_ID"] + top10_keys].to_csv(
    output_dir / "08_top10_candidate_cluster_labels.csv",
    index=False,
    encoding="utf-8-sig"
)

# 11-4. Selected model metadata
selected_model_info = {
    "selected_key": SELECTED_KEY,
    "selected_algorithm": selected_row["algorithm"],
    "selected_feature_set": selected_row["feature_set"],
    "selected_k": int(selected_row["k"]),
    "selected_features": selected_features if selected_features is not None else [],
    "random_state": RANDOM_STATE,
    "selection_reason": [
        "Credit_Capacity_Score를 제외해 고객 신용한도 규모보다 실제 이용 행동에 집중한다.",
        "K-Means는 해석 가능성이 높고 segment profiling 및 발표에 적합하다.",
        "k=6은 계획서의 예상 고객 세그먼트 수와 잘 맞는다.",
        "내부 지표, cluster size balance, 이탈률 차이, high-risk signal 차이, 해석 가능성을 종합적으로 고려하였다."
    ],
    "selected_metrics": {
        key: (
            int(value) if isinstance(value, (np.integer,))
            else float(value) if isinstance(value, (np.floating,))
            else value
        )
        for key, value in selected_row.items()
        if key in [
            "silhouette",
            "davies_bouldin",
            "calinski_harabasz",
            "min_cluster_ratio",
            "max_cluster_ratio",
            "cluster_size_ratio_gap",
            "attrition_rate_range",
            "high_risk_signal_range",
            "selection_score",
        ]
    },
    "selected_model_file": (
        str(selected_model_path.relative_to(ROOT)) if selected_model_path is not None else None
    ),
}

save_json(selected_model_info, output_dir / "09_selected_model.json")

# 11-5. Selected labels
selected_label_df = pd.DataFrame({
    "Row_ID": df_cluster["Row_ID"].values,
    "Cluster": selected_labels,
})

selected_label_df.to_csv(
    output_dir / "10_selected_cluster_labels.csv",
    index=False,
    encoding="utf-8-sig"
)

selected_profile_summary.to_csv(
    output_dir / "10_selected_model_profile_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

# 11-6. Input tables used
df_cluster.to_csv(
    output_dir / "11_df_cluster_input_used.csv",
    index=False,
    encoding="utf-8-sig"
)

df_profile.to_csv(
    output_dir / "12_df_profile_table_used.csv",
    index=False,
    encoding="utf-8-sig"
)

# 11-7. Run metadata
run_metadata = {
    "step": "Step 4 Clustering Model Selection",
    "input_files": {
        "cluster_input": str(cluster_input_path.relative_to(ROOT)),
        "profile_table": str(profile_table_path.relative_to(ROOT)),
    },
    "output_dir": str(output_dir.relative_to(ROOT)),
    "n_rows": int(len(df_cluster)),
    "n_candidates": int(len(all_labels)),
    "candidate_keys": list(all_labels.keys()),
    "top10_keys_by_selection_score": top10_keys,
    "selected_key": SELECTED_KEY,
    "feature_sets": feature_sets,
    "feature_weights": feature_weights,
    "k_range": list(K_RANGE),
    "random_state": RANDOM_STATE,
}

save_json(run_metadata, output_dir / "00_clustering_run_metadata.json")

print("Step 4 outputs saved to:", output_dir.relative_to(ROOT))
print("Saved files:")
for path in sorted(output_dir.iterdir()):
    print("-", path.relative_to(ROOT))

## 12. Step 4 결과 확인

이 셀에서는 최종 선택 모델의 cluster size와 profile summary를 간단히 확인한다.  
깊은 segment 해석은 Step 5에서 수행한다.

In [ ]:
# ------------------------------------------------------------
# 12. Quick check
# ------------------------------------------------------------

cluster_counts = selected_label_df["Cluster"].value_counts().sort_index().rename("n_customers").to_frame()
cluster_counts["ratio"] = cluster_counts["n_customers"] / cluster_counts["n_customers"].sum()

print("Selected model:", SELECTED_KEY)
display(cluster_counts)

print("Selected model profile summary:")
display(selected_profile_summary)

## 13. Git commit 안내

이 노트북 실행 후 다음 파일들이 생성된다.

```text
data/analysis/clustering_results/
```

기본적으로 `data/analysis/`는 재생성 가능한 결과물이므로 `.gitignore`에 의해 commit되지 않을 수 있다.  
제출용으로 결과 파일까지 포함해야 한다면 `.gitignore` 정책을 조정하면 된다.

일반적으로는 노트북만 commit하고, 산출물은 실행을 통해 재생성하는 방식을 추천한다.

```bash
git add notebooks/04_clustering_model_selection.ipynb
git commit -m "Add step 4 clustering model selection notebook"
git push
```